### 例 6.1  
求解下列整数线性规划问题：  

$$
\begin{align*}
\min &\quad z = 40x_1 + 90x_2, \\
\text{s.t.} &\quad 
\begin{cases} 
9x_1 + 7x_2 \leqslant 56, \\
7x_1 + 20x_2 \geqslant 70, \\
x_1, x_2 \geqslant 0 \text{ 为整数}.
\end{cases}
\end{align*}
$$  


**解** 利用 cvxpy 库，求得的最优解为 $ x_1 = 2, x_2 = 3 $；目标函数的最优值为 $ z = 350 $。

In [2]:
import cvxpy as cp
import numpy as np

c = np.array([40, 90])
a = np.array([[9, 7], [-7, -20]])
b = np.array([56, -70])
x = cp.Variable(2, integer=True)  # 定义两个整数(integer)决策变量
obj = cp.Minimize(c @ x)
cons = [a @ x <= b, x >= 0]
prob = cp.Problem(obj, cons)
prob.solve(solver=cp.GLPK_MI, verbose=True)  # 求解并输出求解过程

print(f"最优值为: {prob.value}")
print(f"最优解为:\n{x.value}")


(CVXPY) Aug 25 07:36:41 PM: Your problem has 2 variables, 4 constraints, and 0 parameters.
(CVXPY) Aug 25 07:36:41 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Aug 25 07:36:41 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Aug 25 07:36:41 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Aug 25 07:36:41 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Aug 25 07:36:41 PM: Compiling problem (target solver=GLPK_MI).
(CVXPY) Aug 25 07:36:41 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> GLPK_MI
(CVXPY) Aug 25 07:36:41 PM: Applying reduction Dcp2Cone
(CVXPY) Aug 25 07:36:41 PM: Applying reduction CvxAttr2Constr
(CVXPY) Aug 25 07:36:41 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Aug 25 07:36:41 PM: Applying reduction GLPK_MI
(CVXPY) Aug 25 07:36:41 PM: Finished problem compilation (

                                     CVXPY                                     
                                     v1.7.1                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
最优值为: 350.0
最优解为:
[2. 3.]


### 1. 标准指派问题

### 例 6.2  
某商业公司计划开办 5 家新商店，决定由 5 家建筑公司分别承建。已知建筑公司 $ A_i\ (i = 1,2,3,4,5) $ 对新商店 $ B_j\ (j = 1,2,3,4,5) $ 的建造费用报价（万元）为 $ c_{ij}\ (i,j = 1,2,3,4,5) $，见表 6.1。为节省费用，商业公司应当对 5 家建筑公司怎样分配建造任务，才能使总的建造费用最少？  


#### 表 6.1 建造费用报价数据  

|    | $ B_1 $ | $ B_2 $ | $ B_3 $ | $ B_4 $ | $ B_5 $ |
| ---- | ---- | ---- | ---- | ---- | ---- |
| $ A_1 $ | 4 | 8 | 7 | 15 | 12 |
| $ A_2 $ | 7 | 9 | 17 | 14 | 10 |
| $ A_3 $ | 6 | 9 | 12 | 8 | 7 |
| $ A_4 $ | 6 | 7 | 14 | 6 | 10 |
| $ A_5 $ | 6 | 9 | 12 | 10 | 6 |  


**解** 这是一个标准的指派问题。引进 0-1 变量  
$$
x_{ij} = 
\begin{cases} 
1, & A_i \text{ 承建 } B_j, \\
0, & A_i \text{ 不承建 } B_j,
\end{cases}
\quad i,j = 1,2,\cdots,5.
$$  

则问题的数学模型为  
$$
\begin{align*}
\min &\quad z = \sum_{i=1}^{5} \sum_{j=1}^{5} c_{ij} x_{ij}, \\
\text{s.t.} &\quad 
\begin{cases} 
\sum\limits_{j=1}^{5} x_{ij} = 1, & i = 1,2,\cdots,5, \\
\sum\limits_{i=1}^{5} x_{ij} = 1, & j = 1,2,\cdots,5, \\
x_{ij} = 0 \text{ 或 } 1, & i,j = 1,2,\cdots,5.
\end{cases}
\end{align*}
$$  


利用 cvxpy 库，求得的最优解为  
$$
\begin{bmatrix} 
1 & 2 & 3 & 4 & 5 \\
3 & 2 & 1 & 4 & 5 
\end{bmatrix},
$$  

也就是说，最优指派方案是让 $ A_1 $ 承建 $ B_3 $，$ A_2 $ 承建 $ B_2 $，$ A_3 $ 承建 $ B_1 $，$ A_4 $ 承建 $ B_4 $，$ A_5 $ 承建 $ B_5 $。这样安排能使总的建造费用最少，最小费用为 34 万元。  


In [5]:
import numpy as np
import cvxpy as cp

c = np.array([
    [4, 8,  7, 15, 12],
    [7, 9, 17, 14, 10],
    [6, 9, 12,  8,  7],
    [6, 7, 14,  6, 10],
    [6, 9, 12, 10,  6]       
])
x = cp.Variable((5, 5), integer=True)
obj = cp.Minimize(cp.sum(cp.multiply(c, x)))
con = [0 <= x, x <= 1, cp.sum(x, axis=0, keepdims=True)==1, cp.sum(x, axis=1, keepdims=True)==1]  # x在[0, 1]每行每列相加=1
prob = cp.Problem(obj, con)
prob.solve(solver=cp.GLPK_MI)

print(f"最优值为: {prob.value}")
print(f"最优解为:\n{x.value}")

最优值为: 34.0
最优解为:
[[0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]


### 2. 广义指派模型

### 例 6.3（装箱问题）  
有 7 种规格的包装箱要装到两辆铁路平板车上去。包装箱的宽和高是一样的，但厚度 $ l\ (\text{cm}) $ 及重量 $ w\ (\text{kg}) $ 是不同的，表 6.2 给出了每种包装箱的厚度、重量以及数量，每辆平板车有 $ 10.2\ \text{m} $ 长的地方来装包装箱，载重为 $ 40\ \text{t} $，由于当地货运的限制，对 $ C_5, C_6, C_7 $ 类的包装箱的总数有一个特别的限制：这类箱子所占的空间（厚度）不能超过 $ 302.7\ \text{cm} $。要求给出最好的装运方式。  


#### 表 6.2 各类包装箱数据  

|    | $ C_1 $ | $ C_2 $ | $ C_3 $ | $ C_4 $ | $ C_5 $ | $ C_6 $ | $ C_7 $ |
| ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- |
| $ l/\text{cm} $ | 48.7 | 52.0 | 61.3 | 72.0 | 48.7 | 52.0 | 64.0 |
| $ w/\text{kg} $ | 2000 | 3000 | 1000 | 500 | 4000 | 2000 | 1000 |
| 件数 | 8 | 7 | 9 | 6 | 6 | 4 | 8 |  


**解** 这是 1988 年美国大学生数学建模竞赛 B 题。题中所有包装箱共重 $ 89\ \text{t} $，而两辆平板车一共只能载重 $ 80\ \text{t} $，因此不可能全装下。究竟在两辆车上各装哪些种类箱子且各为多少才合适，必须有评价的标准。这个标准就是遵守题中说明的厚度、重量、件数等方面的约束条件，尽可能多装，而尽可能多装有两种理解：一是尽可能在体积上多装，由于规定是按面包片重叠那样的装法，故等价于尽可能使两辆车上的装箱总厚度尽可能大；二是尽可能在重量上多装，即使得两辆车上的装箱总重量尽可能大。  


设决策变量 $ x_{ij}\ (i = 1,2; j = 1,2,\cdots,7) $ 表示第 $ i $ 辆车上装第 $ j $ 种包装箱的件数，$ l_j, w_j, a_j\ (j = 1,2,\cdots,7) $ 分别表示第 $ j $ 种包装箱的厚度、重量和件数。  


下面先就第一种理解，建立数学模型。  

#### 1. 装箱总厚度最大的模型  
首先考虑约束条件。  

##### (1) 件数限制  
$$
\sum\limits_{i=1}^{2} x_{ij} \leqslant a_j, \quad j = 1,2,\cdots,7.
$$  

##### (2) 长度限制  
$$
\sum_{j=1}^{7} l_j x_{ij} \leqslant 1020, \quad i = 1,2.
$$  

##### (3) 重量限制  
$$
\sum_{j=1}^{7} w_j x_{ij} \leqslant 40000, \quad i = 1,2.
$$  

##### (4) 特殊限制  
$$
\sum_{j=5}^{7} l_j (x_{1j} + x_{2j}) \leqslant 302.7.
$$  

另外变量 $ x_{ij} $ 为整型变量。  


目标函数为  
$$
\max\ z_1 = \sum_{j=1}^{7} l_j (x_{1j} + x_{2j}).
$$  


由此得到问题的数学模型：  
$$
\begin{align*}
\max &\quad z_1 = \sum_{j=1}^{7} l_j (x_{1j} + x_{2j}), \\
\text{s.t.} &\quad 
\begin{cases} 
\sum\limits_{i=1}^{2} x_{ij} \leqslant a_j, & j = 1,2,\cdots,7, \\
\sum\limits_{j=1}^{7} l_j x_{ij} \leqslant 1020, & i = 1,2, \\
\sum\limits_{j=1}^{7} w_j x_{ij} \leqslant 40000, & i = 1,2, \\
\sum\limits_{j=5}^{7} l_j (x_{1j} + x_{2j}) \leqslant 302.7, \\
x_{ij} \geqslant 0 \text{ 且为整数}, & i = 1,2; j = 1,2,\cdots,7.
\end{cases}
\end{align*}
$$  


利用 cvxpy 库，可得到问题的最优解：  
$$
x^* = (x_{ij})_{2 \times 7} = 
\begin{bmatrix} 
4 & 1 & 5 & 3 & 3 & 2 & 0 \\
4 & 6 & 4 & 3 & 0 & 1 & 0 
\end{bmatrix}, \quad z_1 = 2039.4.
$$

In [3]:
import numpy as np
import cvxpy as cp

l = np.array([48.7, 52.0, 61.3, 72.0, 48.7, 52.0, 64.0])
w = np.array([2000, 3000, 1000, 500, 4000, 2000, 1000])
a = np.array([8, 7, 9, 6, 6, 4, 8])
x = cp.Variable((2, 7), integer=True)
obj = cp.Maximize(cp.sum(x @ l))
con = [cp.sum(x, axis=0, keepdims=True) <= a.reshape(1, 7), x @ l <= 1020, x @ w <= 40000, cp.sum(x[:, 4:] @ l[4:]) <= 302.7, x >= 0]
prob = cp.Problem(obj, con)
prob.solve(solver=cp.GLPK_MI, verbose=True)

print(f"最优值: {prob.value}")
print(f"最优解:\n{x.value}")

(CVXPY) Aug 25 08:39:28 PM: Your problem has 14 variables, 26 constraints, and 0 parameters.
(CVXPY) Aug 25 08:39:28 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Aug 25 08:39:28 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Aug 25 08:39:28 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Aug 25 08:39:28 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Aug 25 08:39:28 PM: Compiling problem (target solver=GLPK_MI).
(CVXPY) Aug 25 08:39:28 PM: Reduction chain: FlipObjective -> Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> GLPK_MI
(CVXPY) Aug 25 08:39:28 PM: Applying reduction FlipObjective
(CVXPY) Aug 25 08:39:28 PM: Applying reduction Dcp2Cone
(CVXPY) Aug 25 08:39:28 PM: Applying reduction CvxAttr2Constr
(CVXPY) Aug 25 08:39:28 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Aug 25 08:39:28 PM: Apply

                                     CVXPY                                     
                                     v1.7.1                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------


(CVXPY) Aug 25 08:39:29 PM: Problem status: optimal
(CVXPY) Aug 25 08:39:29 PM: Optimal value: 2.039e+03
(CVXPY) Aug 25 08:39:29 PM: Compilation took 1.455e-02 seconds
(CVXPY) Aug 25 08:39:29 PM: Solver (including time spent in interface) took 5.962e-01 seconds


-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
最优值: 2039.4
最优解:
[[4. 1. 5. 3. 3. 2. 0.]
 [4. 6. 4. 3. 0. 1. 0.]]


### 2. 装箱总重量最大的模型  
要使两辆平板车的装箱总重量之和最大，目标函数为  
$$
\max\ z_2 = \sum_{j=1}^{7} w_j (x_{1j} + x_{2j}).
$$  

约束条件与前述模型相同。利用 cvxpy 库，可得到问题的最优解：  
$$
x^* = (x_{ij})_{2 \times 7} = 
\begin{bmatrix} 
6 & 0 & 0 & 6 & 6 & 0 & 0 \\
2 & 7 & 9 & 0 & 0 & 0 & 0 
\end{bmatrix}, \quad z_2 = 73000.
$$

In [4]:
import numpy as np
import cvxpy as cp

l = np.array([48.7, 52.0, 61.3, 72.0, 48.7, 52.0, 64.0])
w = np.array([2000, 3000, 1000, 500, 4000, 2000, 1000])
a = np.array([8, 7, 9, 6, 6, 4, 8])
x = cp.Variable((2, 7), integer=True)
obj = cp.Maximize(cp.sum(x @ w))
con = [cp.sum(x, axis=0, keepdims=True) <= a.reshape(1, 7), x @ l <= 1020, x @ w <= 40000, cp.sum(x[:, 4:] @ l[4:]) <= 302.7, x >= 0]
prob = cp.Problem(obj, con)
prob.solve(solver=cp.GLPK_MI, verbose=True)

print(f"最优值: {prob.value}")
print(f"最优解:\n{x.value}")

(CVXPY) Aug 25 08:41:13 PM: Your problem has 14 variables, 26 constraints, and 0 parameters.
(CVXPY) Aug 25 08:41:13 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Aug 25 08:41:13 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Aug 25 08:41:13 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Aug 25 08:41:13 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Aug 25 08:41:13 PM: Compiling problem (target solver=GLPK_MI).
(CVXPY) Aug 25 08:41:13 PM: Reduction chain: FlipObjective -> Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> GLPK_MI
(CVXPY) Aug 25 08:41:13 PM: Applying reduction FlipObjective
(CVXPY) Aug 25 08:41:13 PM: Applying reduction Dcp2Cone
(CVXPY) Aug 25 08:41:13 PM: Applying reduction CvxAttr2Constr
(CVXPY) Aug 25 08:41:13 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Aug 25 08:41:13 PM: Apply

                                     CVXPY                                     
                                     v1.7.1                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
最优值: 73000.0
最优解:
[[6. 0. 0. 6. 6. 0. 0.]
 [2. 7. 9. 0. 0. 0. 0.]]
